# Linear SVM

Support Vector Machines are linear classifiers that seek maximum-margin separating hyperplanes.

The primal SVM objective is the minimization of:

$$J(w, b) = \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{n}\max(0, 1 - z_i(w^T x_i + b))$$

where $n$ is the number of training samples, $z_i \in \{-1, +1\}$ are the labels, and $C > 0$ is the penalty parameter that controls the trade-off between margin maximization and misclassification tolerance.

The **dual formulation** is obtained by introducing Lagrange multipliers $\alpha_i$ and is expressed as the maximization of:

$$J^D(\alpha) = -\frac{1}{2}\alpha^T H_c \alpha + \alpha^T \mathbf{1}$$

$$\text{s.t. } 0 \leq \alpha_i \leq C, \quad \sum_{i=1}^{n} \alpha_i z_i = 0$$

where $H_c^{i,j} = z_i z_j x_i^T x_j$.

Once the optimal $\alpha^*$ is found, the primal parameters are recovered as:

$$w^* = \sum_{i=1}^{n} \alpha_i^* z_i x_i, \quad b^* \text{ from KKT conditions}$$

and the classification score for a new sample $x$ is:

$$w^{*T}x = \sum_{i=1}^{n}\alpha_i^* z_i k(x_i, x) = \sum_{i|\alpha_i \neq 0}\alpha_i^* z_i k(x_i, x)$$

For linear SVMs, we can optimize either the primal or dual formulation. Unfortunately, the unconstrained primal objective is not differentiable. Although L-BFGS may still find the minimizer of $J$, there is no guarantee that the algorithm will converge close to the optimal $(w, b)$. Specialized solvers exist but are outside the scope of this course.

The dual formulation is differentiable, but it contains both box constraints ($a \leq \alpha_i \leq b$) and the additional equality constraint $\sum \alpha_i z_i = 0$. The L-BFGS algorithm can handle box constraints but cannot incorporate the equality constraint.

The constraint $\sum \alpha_i z_i = 0$ arises from the bias term in the primal SVM formulation. We therefore modify the SVM problem slightly to eliminate this constraint, allowing us to use L-BFGS-B (B stands for box-constraints).

We reformulate the primal as minimization of:

$$J_b(w_b) = \frac{1}{2}\|w_b\|^2 + C\sum_{i=1}^{n}\max(0, 1 - z_i(w_b^T x_b^i))$$

where:

$$x_b^i = \begin{pmatrix} x_i \\ 1 \end{pmatrix}, \quad w_b = \begin{pmatrix} w \\ b \end{pmatrix}$$

We observe that $w_b^T x_b^i = w^T x + b$, i.e. the scoring rules have the same expression as the original formulation. However, unlike the original SVM, we are also regularizing the bias term, since we regularize the norm of $w_b$:

$$\|w_b\|^2 = \|w\|^2 + b^2$$

Regularizing the bias may, in general, lead to sub-optimal margin decisions. We can mitigate this effect using a mapping:

$$x_b^i = \begin{pmatrix} x_i \\ K \end{pmatrix}$$

As $K$ grows larger, the regularization effect on $b$ becomes weaker; however, the algorithm may require more iterations to converge.

The dual objective of the modified primal becomes the maximization of:

$$J_b^D(\alpha) = -\frac{1}{2}\alpha^T H_c\alpha + \alpha^T \mathbf{1}$$

$$\text{s.t. } 0 \leq \alpha_i \leq C, \quad \forall i \in \{1 \ldots n\}$$

i.e., the same formulation as before but without the equality constraint, and with the matrix $H_c$ computed from the extended features $x_b^i$ rather than from the original features $x_i$:

$$H_c^{i,j} = z_i z_j x_b^{i\,T} x_b^j$$

> See implementation in [`scripts/SVM.py`](../scripts/SVM.py): function `train_SVM` (dual linear SVM with extended features).
> Dataset and evaluation: [`scripts/utils.py`](../scripts/utils.py), [`scripts/bayes_error.py`](../scripts/bayes_error.py).

In [11]:
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy.optimize 
import sklearn
from scripts.utils import *
from scripts.SVM import *
from scripts.logisticreg import *
from scripts.bayes_error import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
D, L = load_iris_binary()
(DTR, LTR), (DVAL, LVAL) = split_db_2to1(D, L)


In [13]:
alpha,w, b = train_SVM(1, 0.1, DTR, LTR) #esempio di funzionamento

Primal Loss : 3.774973825264006 - Dual Loss : 3.774973824823999


In [14]:
KC = np.array([
    [1.0, 0.1],
    [1.0, 1.0],
    [1.0, 10.0],
    [10.0, 0.1],
    [10.0, 1.0],
    [10.0, 10.0]
])

pi_T=0.5
for kc in KC:
    K=kc[0]
    C=kc[1]

    print(f"================================= K: {K} C: {C}==============================================================")
    alpha,w,b=train_SVM(K,C,DTR,LTR)
    s=w.T @ DVAL + b
    predictions=s >0
    error_rate=np.mean(predictions != LVAL)
    min_dcf=compute_min_dcf(s,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(s, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)


    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

    print(f"==================================================================================================")



================================= K: 1.0 C: 0.1==============================================================
Primal Loss : 3.774973825264006 - Dual Loss : 3.774973824823999
Error Rate: 2.9%
DCF min: 0.05555555555555555
Actual DCF:0.0625

================================= K: 1.0 C: 1.0==============================================================
Primal Loss : 15.779935357361582 - Dual Loss : 15.779930849554196
Error Rate: 5.9%
DCF min: 0.05555555555555555
Actual DCF:0.11805555555555555

================================= K: 1.0 C: 10.0==============================================================
Primal Loss : 78.96899183534113 - Dual Loss : 78.96892834899586
Error Rate: 5.9%
DCF min: 0.05555555555555555
Actual DCF:0.11805555555555555

================================= K: 10.0 C: 0.1==============================================================
Primal Loss : 2.9835762584584278 - Dual Loss : 2.9835762584584327
Error Rate: 11.8%
DCF min: 0.0
Actual DCF:0.2222222222222222

===============

# Kernel SVM

SVMs enable non-linear classification through implicit feature expansion into a higher-dimensional space. The dual SVM objective depends on training samples only through dot products, and we can compute a classification score via dot products between training and test samples. Therefore, the SVM does not require us to explicitly compute the feature expansion: it suffices to be able to compute the dot product between expanded features $k(x_1, x_2) = \varphi(x_1)^T \varphi(x_2)$. The function $k$ is called the **kernel function**.

We implement the dual kernel SVM (without bias) for generic kernel functions. The code developed earlier is reused, but we replace the computation of $H_c$ with:

$$H_c^{i,j} = z_i z_j k(x_i, x_j)$$

Unlike the linear SVM, we cannot directly compute the primal solution and its cost. However, we can exploit the primal-dual relationship to express the primal objective in terms of $\alpha$:

$$J_b(w_b) = \frac{1}{2}\|w_b\|^2 + C\sum_{i=1}^{n}\max(0, 1 - z_i(w_b^T x_b^i)) = \frac{1}{2}\alpha^T H \alpha + C\sum_{i=1}^{n}\max(0, 1 - H_{i\cdot}\alpha)$$

where $H_{i\cdot}$ is the $i$-th row of $H$. Note that the terms $H_{i\cdot}\alpha$ correspond to the components of the vector $H\alpha$.

The score of a test sample is:

$$s(x_t) = \sum_{i=1}^{n}\alpha_i^* z_i k(x_i, x_t)$$

where the sum runs over training samples (in practice, only those with $\alpha_i > 0$, i.e. the support vectors).

Available kernels:

- **Polynomial kernel of degree $d$**: $k(x_1, x_2) = (x_1^T x_2 + c)^d$
- **Radial Basis Function (RBF) kernel**: $k(x_1, x_2) = e^{-\gamma\|x_1 - x_2\|^2}$

Kernel and hyperparameter selection (e.g. $c$, $\gamma$) can also be performed through cross-validation.

**NOTE**: To add a (regularized) bias in the non-linear SVM, it is not sufficient to simply extend the feature vectors as done previously. Instead, we add a constant value to the kernel function:

$$\tilde{k}(x_1, x_2) = k(x_1, x_2) + \xi$$

> See implementation in [`scripts/SVM.py`](../scripts/SVM.py): functions `train_kernel_poly_SVM`, `train_kernel_KRB`, `polygrad_d`, `KRB`, `regular_kernel`.

In [23]:

KC_poly = np.array([
    [0.0, 1.0,2,0],
    [1.0, 1.0,2,0],
    [0.0, 1.0,2,1],
    [1.0,1.0,2,1],
])
KC_KRB = np.array([
    [0.0, 1.0,1],
    [1.0, 1.0,1],
    [0.0, 1.0,10],
    [1.0,1.0,10],
])
pi_T=0.5
ZTR=LTR*2-1
for kc in KC_poly:
    K=kc[0]
    C=kc[1]
    d=kc[2]
    c=kc[3]
    xi=K**2

    print(f"================================= K: {K} C: {C} Poly (d={d},c={c})==============================================================")
    alpha=train_kernel_poly_SVM(K,C,DTR,LTR,xi,d,c)
    reg_kernel=regular_kernel(polygrad_d(DTR,d,DVAL,c),xi)
    s=alpha*ZTR @ reg_kernel
    predictions=s >0
    error_rate=np.mean(predictions != LVAL)
    min_dcf=compute_min_dcf(s,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(s, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)


    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

    print(f"==================================================================================================")



================================= K: 0.0 C: 1.0 Poly (d=2.0,c=0.0)==============================================================
Primal Loss : 6.66363396877415 - Dual Loss : 6.663627830941669
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.1736111111111111

================================= K: 1.0 C: 1.0 Poly (d=2.0,c=0.0)==============================================================
Primal Loss : 6.296917447649182 - Dual Loss : 6.296912183693136
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.1736111111111111

================================= K: 0.0 C: 1.0 Poly (d=2.0,c=1.0)==============================================================
Primal Loss : 3.5929195345538525 - Dual Loss : 3.5929184422642098
Error Rate: 2.9%
DCF min: 0.05555555555555555
Actual DCF:0.05555555555555555

================================= K: 1.0 C: 1.0 Poly (d=2.0,c=1.0)==============================================================
Primal Loss : 3.570001605180356 - Dual Loss : 3.5699865797767165
Error Rate: 2.9%
DCF

In [24]:

for kc in KC_KRB:
    K=kc[0]
    C=kc[1]
    gamma=kc[2]
    xi=K**2

    print(f"================================= K: {K} C: {C} KRB (d={d},c={c})==============================================================")
    alpha=train_kernel_KRB(K,C,DTR,LTR,xi,gamma)
    reg_kernel=regular_kernel(KRB(DTR,DVAL,gamma),xi)
    s=alpha*ZTR @ reg_kernel
    predictions=s >0
    error_rate=np.mean(predictions != LVAL)
    min_dcf=compute_min_dcf(s,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(s, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)


    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

    print(f"==================================================================================================")

================================= K: 0.0 C: 1.0 KRB (d=2.0,c=1.0)==============================================================
Primal Loss : 11.989317276783364 - Dual Loss : 11.989298414692115
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.1736111111111111

================================= K: 1.0 C: 1.0 KRB (d=2.0,c=1.0)==============================================================
Primal Loss : 11.978144832482638 - Dual Loss : 11.978133312353595
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.1736111111111111

================================= K: 0.0 C: 1.0 KRB (d=2.0,c=1.0)==============================================================
Primal Loss : 18.4276236479088 - Dual Loss : 18.427568671787714
Error Rate: 11.8%
DCF min: 0.1736111111111111
Actual DCF:0.22916666666666666

================================= K: 1.0 C: 1.0 KRB (d=2.0,c=1.0)==============================================================
Primal Loss : 18.391862918160726 - Dual Loss : 18.39181622127977
Error Rate: 8.8%
DCF 